[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q2/01_orientation_and_precommit.ipynb)

# R2-Q2 Week 1 — Orient on the failure-mode literature and commit to your taxonomy

This notebook orients you on the failure-mode literature that R2-Q2 builds on — Adebayo et al. (2018) on saliency-method sanity checks, Noyan (2022) on PlantVillage background shortcuts, and the Grad-CAM mechanics you'll use to inspect misclassifications — and recaps the R2-Q1 outputs your work here inherits, including the headline finding that PlantVillage-trained accuracy on PlantDoc drops from 94.6% on fungal categories to 48.9% on healthy. The Week 1 deliverable is your `precommit.json` recording three written commitments: a five-category failure-mode taxonomy with decision rules, a categorization procedure, and the aggregation level you'll report at.

By the end of this notebook you will have:

- A `precommit.json` file recording your five-category taxonomy (with an "other / ambiguous" bucket), your categorization procedure, and your aggregation level — locked in writing before you see any Grad-CAM output.
- A documented understanding of the R2-Q1 outputs this question inherits: the trained classifier (`baseline_resnet18.pt`), the PD prediction table (`pd_predictions.parquet`, 236 rows), and the class-space mappings from R2-Q1's precommit.
- The literature context for Week 2's load-and-filter work and Week 3's Grad-CAM + sanity-check work: which methods pass Adebayo's checks, what shortcut learning looks like in PlantVillage specifically, and what "background-attended" failure modes have been observed in prior work.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in the following Cell:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in the following Cell:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive and set up irilab2026.
import irilab2026 as iri
iri.setup(gpu_required=False)

OUTPUT_DIR = iri.output_dir("r2_q2")
print(f"Output directory: {OUTPUT_DIR}")

## 2) What R2-Q2 inherits from R2-Q1

R2-Q1 measured the gap between lab and field conditions for plant disease classification. It trained a ResNet-18 classifier on PlantVillage — lab-quality, controlled-lighting, single-leaf-on-uniform-background images — and applied that trained model to PlantDoc, which is field photographs of the same diseases in real-world conditions: variable lighting, natural backgrounds, occluding objects, multiple leaves per image.

R2-Q1's headline finding: PlantVillage-trained accuracy on PlantDoc drops from 94.6% on fungal categories to 48.9% on healthy. The transfer failure isn't uniform — the model holds nearly all of its accuracy on one category and loses roughly half of it on another. *Where* the gap lives is settled. *What mechanism* produces it isn't.

That mechanism question is what R2-Q2 picks up. You'll run Grad-CAM on each PD misclassification, categorize the resulting attention heatmap into a pre-committed failure-mode taxonomy, and report the distribution across the five categories: background-attended, leaf-shape-attended, lighting-attended, symptom-attended-but-wrong-class, and other/ambiguous. The taxonomy itself, and the decision rules that operationalize it, are what you commit to in this notebook.

The R2-Q1 → R2-Q2 narrative bridge — *does the failure-mode distribution within healthy-misclassified images differ from the overall distribution?* — is held for the paper, where both questions' results can be discussed together. What's surfaced here is just the inheritance: R2-Q1 produced the predictions you'll inspect, the classifier you'll inspect them with, and the class-space mapping you'll use to translate model outputs into category labels.

### The four files you inherit

You don't load any of these in this notebook — N02 (Week 2) is where the loading happens. The inventory here is so you know what to expect when you open N02:

| File (in `iri.output_dir("r2_q1")`) | What it is | How N02–N04 use it |
|---|---|---|
| `baseline_resnet18.pt` | The trained PlantVillage classifier | N03 loads it for Grad-CAM passes and for the model-parameter randomization sanity check |
| `pd_predictions.parquet` | The 236-row PlantDoc prediction table | N02 filters to `correct_at_category == False`; N03 reads the model's predicted class per image as the Grad-CAM target |
| `precommit.json` | R2-Q1's Week 1 precommit | N02 pulls `pv_class_to_category` and `pd_class_to_category` for class-name lookups |
| `pv_predictions.parquet` | The PlantVillage-internal prediction table (reference) | Not strictly needed for R2-Q2; available if you want to cross-check PV-side predictions |

If you ran R2-Q1 yourself last week, these files are already in your `r2_q1` output directory. If you're starting R2-Q2 without having done R2-Q1, you'll need to run the R2-Q1 notebook series first — the R2-Q2 workflow assumes those files exist, and N02's first move is to load them.

## 3) The failure-mode literature — one paragraph per paper

Before you commit to your taxonomy in Section 4, three papers anchor what we know about the kind of failures you'll be inspecting and the tool you'll be inspecting them with. The treatment here is intentionally light — one paragraph per paper, just enough orientation to ground the pre-commit decisions in Sections 5–10. The papers are Adebayo et al. (2018) on whether attention-visualization methods can be trusted, Noyan (2022) on how PlantVillage models latch onto backgrounds, and Selvaraju et al. (2017) on the Grad-CAM mechanics themselves.

### 3.1) Adebayo et al. (2018) — Sanity checks for saliency maps

Saliency methods produce heatmaps over an input image showing which pixels contributed most to a model's prediction — they're the tool we use when we want to see what a model was "looking at." Adebayo et al. showed that several popular saliency methods produce visually plausible heatmaps even when applied to fully randomized models, meaning the heatmaps were reading patterns in the input image rather than what the model had learned. To separate real model attention from input-image patterns, they introduced two sanity checks: **model-parameter randomization** progressively replaces the trained model's weights with random values from the top layers backward and confirms that the saliency map diverges from the trained-model map as randomization proceeds; **data randomization** trains a model from scratch on label-shuffled data and confirms that the resulting saliency map differs from the real model's. A method that passes both is reading what the model actually learned. Vanilla Grad-CAM — the variant you'll use in N03 — passes both checks on most architectures Adebayo et al. tested, but that doesn't exempt you from running the checks on your specific trained model: Section 10 is where you commit to the sanity-check procedure, and N03 is where you execute it.

### 3.2) Noyan (2022) — Uncovering bias in the PlantVillage dataset

Noyan tested whether PlantVillage class labels could be predicted from the background alone — the regions of each image that aren't the leaf. By masking the leaves out and training classifiers on the remaining background pixels, Noyan found that PV labels remained predictable well above chance: even with the background downsampled to just eight pixels, classifiers reached around 49% accuracy on the 38-class task (chance would be roughly 3%). This is the empirical basis for the "background-attended" category in your taxonomy. If PV-trained models are learning background regularities that correlate with labels, those regularities are part of what the model uses to predict — and field photographs in PlantDoc don't share those regularities, which would explain a transfer failure concentrated in the conditions where the background changes most. Whether your specific trained classifier from R2-Q1 actually attended to backgrounds when it failed on PD is what Sections 4–8 of this notebook set you up to measure.

### 3.3) Selvaraju et al. (2017) — Grad-CAM

Selvaraju et al. introduced Gradient-weighted Class Activation Mapping (Grad-CAM): a method for producing a class-discriminative attention heatmap from any convolutional network by combining the gradients of a chosen class's score with the spatial activations of a chosen convolutional layer. Mechanically: take the gradient of the predicted-class score with respect to each feature map in the target layer, average those gradients over each map to get a per-map weight, take a weighted sum of the feature maps using those weights, apply ReLU to keep only positive contributions, and the result is a low-resolution heatmap showing which image regions most supported the chosen class. For ResNet-18 with 224×224 inputs and the last convolutional block (`layer4`) as the target, the heatmap comes out at 7×7 spatial resolution before being upsampled to the input size — coarse enough that "the model attended to the upper-left region" is the right grain of conclusion, not "the model attended to pixel (37, 84)." This is the tool you'll use in N03 to produce one heatmap per PD misclassification, and those heatmaps are the input that N04 categorizes against your pre-committed taxonomy.

## 4) Pre-commit your taxonomy

The Week-1 pre-commit discipline is straightforward: before you see any Grad-CAM output, you write down what you're going to do with the output. That includes the categories you'll sort failures into, the decision rules that operationalize each category, the aggregation level you'll report the distribution at, and the criterion your sanity checks have to meet. A taxonomy chosen after seeing the data is a taxonomy fit to the data — and the "the model fails in recognizable patterns" finding isn't a real test if the patterns were derived from the same images you're analyzing.

The four pre-commits become four top-level fields in `precommit.json`:

- **`taxonomy`** — the categories themselves (this section, Section 4).
- **`categorization_procedure`** — the rules for assigning each Grad-CAM heatmap to a category (Sections 5–8, one section per sub-decision).
- **`aggregation_level`** — what level you report the distribution at (Section 9).
- **`sanity_checks`** — the criterion your saliency method has to meet, per Adebayo (Section 10).

This section commits the first one and initializes the dict.

The taxonomy has five categories: four named, plus an "other / ambiguous" bucket. Each named category corresponds to a failure mode that prior work has either directly documented or that PlantVillage's data-collection conventions make plausible:

- **`background_attended`** — the model is looking at the background rather than the leaf. Grounded in Noyan (2022)'s finding that PV labels are predictable from the background alone at well above chance.
- **`leaf_shape_attended`** — the model is looking at the leaf outline rather than disease symptoms. PV's uniform single-leaf framing means leaf shape correlates with host species, and host species correlates with disease in PV's class structure — so leaf shape becomes a shortcut for disease identity.
- **`lighting_attended`** — the model is looking at lighting conditions. PV is studio-lit; PD has natural and variable lighting. If lighting co-varied with category in the training data, the model could be reading lighting cues that don't generalize to field photographs.
- **`symptom_attended_but_wrong_class`** — the model is looking at a real symptom on the leaf but assigning it to the wrong disease. This is the "honest mistake" category: the model engaged with the leaf and the lesion, but its class boundary was wrong.
- **`other_ambiguous`** — none of the above, or more than one of the above. The question page requires this category explicitly, as protection against forcing every failure into a prepared box. A non-trivial share of failures landing here is itself a result: it means the taxonomy as committed isn't capturing the actual structure of the failures, and that's a finding worth reporting in the paper.

The decision rules that operationalize these categories — the numeric thresholds, the predicate order, the leaf-segmentation method — don't live in this section. They get built up in Sections 5–8 under the `categorization_procedure` field. What you commit here is the *list* of categories; the *rules* are downstream.

In [ ]:
# Initialize the precommit dict with its four top-level fields.
# Each one will be filled in by the section indicated:
#   - taxonomy:                 this section (Section 4)
#   - categorization_procedure: Sections 5-8 (one sub-field per section)
#   - aggregation_level:        Section 9
#   - sanity_checks:            Section 10
precommit = {
    "taxonomy": None,
    "categorization_procedure": {},   # empty dict so Sections 5-8 can add sub-fields
    "aggregation_level": None,
    "sanity_checks": None,
}

# Commit the five-category taxonomy.
precommit["taxonomy"] = {
    "categories": [
        "background_attended",
        "leaf_shape_attended",
        "lighting_attended",
        "symptom_attended_but_wrong_class",
        "other_ambiguous",
    ],
    "decision_rules_pointer": "see categorization_procedure",
    "reasoning": (
        # Replace this placeholder with your own reasoning before submission.
        # Cover at minimum: why these five categories (not four or six), how
        # the four named categories map to documented failure modes in prior
        # work, and what an unexpectedly large "other_ambiguous" share would
        # tell you about whether the taxonomy fits the actual failures.
        "PLACEHOLDER — REWRITE BEFORE SUBMISSION."
    ),
}

# Print back what just got committed.
print(f"Taxonomy committed: {len(precommit['taxonomy']['categories'])} categories")
for cat in precommit["taxonomy"]["categories"]:
    print(f"  - {cat}")

## 5) Pre-commit your leaf-segmentation method

Before any sub-decision in this section, the umbrella commitment for Sections 5–8: you're operationalizing the taxonomy by image processing, not by human raters. The question page recommends image processing for the solo five-week scope — no second rater is available to compute inter-rater agreement, and image processing turns each category from a prose definition into a numeric rule against pixel-level quantities derived from the Grad-CAM heatmap. The cost is that each sub-decision across Sections 5–8 has to be defended explicitly: the segmentation method, the attention-mass definition, the predicate scheme and order, and the threshold values are all going to drive your eventual category assignments, and a reader of your methods section should be able to reproduce them from the precommit alone.

Section 5 commits the first sub-decision: how you extract the leaf region from each PD image so that "is the model attending to the leaf or to the background?" can be answered numerically. Three commitments in this section:

- **§ 5.1** — the segmentation method and prompting strategy: SAM with a single foreground point at the image center.
- **§ 5.2** — the validation procedure: five hand-segmented images compared against SAM via IoU, interpreted as a sanity check.
- **§ 5.3** — the fallback if SAM fails on a non-trivial share of images: color-based thresholding, with the bias documented in methods.

### 5.1) The segmentation method and prompting strategy

You'll use the Segment Anything Model (SAM) — Meta AI's general-purpose segmentation model — with a single foreground point at each image's center as the prompt. The choice has two parts.

The first part is SAM itself. Leaf segmentation has been done with color thresholding (HSV-space green-pixel masks), with classical computer-vision algorithms (GrabCut, watershed), and with leaf-specific neural networks trained on labeled datasets. Each has tradeoffs: color thresholding is fast and dependency-light but fails on diseased leaves where the foliage isn't uniformly green (chlorosis, necrosis, severe lesions); classical algorithms need careful per-image tuning; leaf-specific networks require labeled training data we don't have for PD. SAM is the pragmatic choice for a five-week scope — pretrained, prompt-driven, and robust across visual conditions including unhealthy leaves. The cost is heavier compute (SAM is a transformer), manageable on Colab GPU for the size of PD's misclassification set; this is part of why Section 5's actual segmentation work is deferred to N04 rather than attempted in this orientation notebook.

The second part is the prompting strategy. SAM takes a prompt — a point, a box, or a mask hint — and segments the region containing that prompt. For PD images, the leaf is usually but not always the dominant object near the image center. A single foreground point at the center captures the common case cheaply and reproducibly; box prompts would require pre-localizing the leaf (which is the problem we're trying to solve), and automatic mask generation would return multiple candidate masks and require a selection rule. The single center-point prompt is a deliberately simple default — it works for the majority of PD images and produces an obvious failure case (off-center leaves, multi-leaf images) that the validation in § 5.2 is designed to surface.

The locked commitment for this sub-decision: `method = "SAM (Segment Anything Model)"`, `prompting = "single point at image center"`.

### 5.2) The validation procedure

You'll validate the SAM masks by hand-segmenting five PD images and computing IoU (intersection-over-union) between your hand-segmented masks and SAM's masks. Three sub-commitments worth being explicit about.

**The number of images is five.** Not three (too few for any pattern to surface), not twenty (too much hand work for a Week-4 task with N04's other demands). Five is a pragmatic small-sample size — enough to spot a systematic failure mode (SAM consistently picking the wrong object, or consistently undersegmenting, say), too few to estimate SAM's precision on PD with any confidence. The five images are sampled with a fixed random seed in N04 so the validation is reproducible.

**The metric is IoU.** Intersection-over-union of your hand-drawn mask with SAM's mask, computed per image. IoU is the standard segmentation-comparison metric, ranges 0 (no overlap) to 1 (identical masks), and is well-understood in the computer-vision literature. Reporting per-image IoU plus the mean across the five is what your methods section will carry.

**The interpretation is sanity check, not precision estimate.** This is the most important commitment of the three. Five images is too few to estimate SAM's PD-wide precision; the validation is a *check* on whether SAM's outputs are anywhere near reasonable, not a *measurement* of how reasonable they are. Your methods section should say something like "mean IoU was X across n=5 hand-validated images; this is a sanity check, not a precision estimate" — and your interpretation of any predicate that depends on SAM masks (background-attended, leaf-shape-attended) should carry that uncertainty forward.

The hand-segmentation work itself happens in N04 (Week 4). Section 3.2 of N04 opens with a markdown cell framing the work as the execution of this Week-1 commitment. What you're committing to here is the procedure: five images, IoU, sanity-check interpretation.

### 5.3) The fallback if SAM fails

If the validation in § 5.2 reveals that SAM is failing on a non-trivial share of images, the fallback is color-based thresholding: an HSV-space green-pixel mask. The threshold values would be tuned per-batch as a methodology adjustment, and the bias would be documented explicitly in your methods section — color thresholding fails predictably on chlorotic, necrotic, or severely lesioned leaves, which are exactly the leaves a disease classifier sees most.

"Non-trivial share" isn't pre-specified as a number here — it's a judgment call you make in N04 after seeing the IoU results. A single bad image among five doesn't trigger the fallback; mean IoU below 0.5 across the five probably does; a clear pattern (SAM picking the wrong object on every off-center leaf, say) definitely does. The reasoning your methods section eventually carries should explain the threshold you applied to the decision, not just whether you took the fallback.

The fallback exists as an honest pre-commit, not as expected behavior. If SAM works reasonably on PD — which prior published work on plant imagery suggests it should — the fallback never activates. Committing to it now means that if SAM does fail, you have a documented alternative that doesn't require improvising under time pressure in Week 4.

In [ ]:
# Commit the umbrella categorization method for Sections 5-8.
# You're operationalizing the taxonomy by image processing (numeric rules
# against pixel-level quantities) rather than by human raters. The
# sub-decisions in Sections 5-8 work out the operationalization.
precommit["categorization_procedure"]["method"] = "image_processing"

# Commit the leaf-segmentation sub-decision (sub-sections 5.1 - 5.3).
precommit["categorization_procedure"]["leaf_segmentation"] = {
    "method": "SAM (Segment Anything Model)",
    "prompting": "single point at image center",
    "validation": {
        "approach": "hand-segmented sample, IoU against SAM masks",
        "n_validation_images": 5,
        "iou_interpretation": "sanity check, not precision estimate",
    },
    "fallback_if_sam_fails": "color-based thresholding, bias documented in methods",
}

# Reasoning placeholder for the whole categorization_procedure field.
# This single field covers the umbrella image-processing choice and all
# four sub-decisions across Sections 5-8. Sections 6, 7, and 8 will add
# their own sub-fields without re-setting this placeholder.
precommit["categorization_procedure"]["reasoning"] = (
    # Replace this placeholder with your own reasoning before submission.
    # Cover at minimum: why image-processing rather than human raters; the
    # SAM-with-center-point choice and what it does well vs poorly; the
    # validation procedure's role as a sanity check (not a precision
    # estimate); how the four threshold values from Section 8 trade false
    # positives against false negatives; and what the predicate order in
    # Section 7 says about which failure modes you prioritize when more
    # than one fits.
    "PLACEHOLDER — REWRITE BEFORE SUBMISSION."
)

# Print back what just got committed.
ls = precommit["categorization_procedure"]["leaf_segmentation"]
print(f"Categorization method:    {precommit['categorization_procedure']['method']}")
print(f"Segmentation method:      {ls['method']}")
print(f"Prompting strategy:       {ls['prompting']}")
print(f"Validation:               n={ls['validation']['n_validation_images']}, "
      f"{ls['validation']['approach']}")
print(f"IoU interpretation:       {ls['validation']['iou_interpretation']}")
print(f"Fallback if SAM fails:    {ls['fallback_if_sam_fails']}")
print("Reasoning placeholder set at categorization_procedure level.")

## 6) Pre-commit your attention-mass definition

The Grad-CAM heatmap you produce in N03 is a 7×7 array of values — gradient-weighted activations from `layer4` of the ResNet-18 classifier (refresher in § 3.3). Each value indicates how strongly that spatial location contributed to the predicted class, but the raw values aren't directly usable for the predicates Section 7 commits to. A predicate like "more than 50% of the model's attention falls outside the leaf" needs an "attention" quantity that's well-defined and comparable across images: non-negative (so "more than 50%" makes sense), and on a fixed scale (so an image with a strong overall signal doesn't dominate the cross-image comparison just by having larger raw values).

This section commits to one composite transform that turns the raw 7×7 array into that quantity. The transform has two operations applied in sequence, and the resulting per-pixel values are what Section 8's threshold definitions and Section 7's predicates read against.

### 6.1) The committed transform: **apply ReLU, then normalize so the values sum to one.**

**Why ReLU first.** The raw Grad-CAM array has both positive and negative values. Positive values indicate spatial locations whose activations supported the predicted class; negative values indicate locations that pulled the prediction *away* from the predicted class. For categorizing where the model "looked" when it made the wrong prediction, what matters is where the supporting evidence was concentrated — the positive contributions. ReLU zeros out the negative values, which is also the standard convention in Selvaraju et al.'s original Grad-CAM definition. Without it, an alternative like absolute-value would conflate "the model was paying attention here in favor of the predicted class" with "the model was paying attention here against the predicted class," which are categorically different things and would muddy every downstream predicate.

**Why normalize to sum to one.** After ReLU, the per-pixel values are non-negative but their absolute magnitudes vary widely across images — some heatmaps have one dominant peak, others are diffuse. Without normalization, "the model's attention mass outside the leaf" isn't a comparable quantity across images: an image with a strong overall signal might have more attention outside the leaf than a low-signal image even if a smaller *fraction* of its attention falls out there. Normalizing so the per-pixel values sum to 1 across the whole heatmap turns the array into a probability-like distribution over pixels — each value is the fraction of total attention mass at that location — and "X% of mass falls outside the leaf" becomes a meaningful question with a comparable answer across images.

The committed definition gets stored as a single string in `precommit.json` to keep the precommit human-readable. The actual computation — apply ReLU to the heatmap, then divide by the sum — lives in N04's per-image processing loop, with the precommit string cited in the methods section as the operational definition. The upsampling step (Grad-CAM heatmaps are saved at the native 7×7 resolution in N03 and upsampled per image at use time in N04) is an implementation detail, not a methodology commitment — what's committed here is the ReLU + normalize transform, regardless of which resolution the heatmap is in when the transform is applied.

In [ ]:
# Commit the attention-mass definition (Section 6).
precommit["categorization_procedure"]["attention_mass_definition"] = "ReLU-then-normalize-to-sum-to-one"

# Print back what just got committed.
print(f"Attention-mass definition: {precommit['categorization_procedure']['attention_mass_definition']}")

## 7) Pre-commit your predicate scheme and order

A predicate is a true/false rule defined over the derived quantities you'll compute from each Grad-CAM heatmap in N04. The four named categories from your taxonomy each get a predicate: a rule for whether a given heatmap counts as background-attended, leaf-shape-attended, lighting-attended, or symptom-attended-but-wrong-class. The derived quantities themselves (fraction of attention mass outside the leaf, fraction in a perimeter band, correlation with brightness, concentration of in-leaf attention) and the threshold values that define each predicate are committed in Section 8.

This section commits two decisions about how the four predicates are combined when applied to a single heatmap:

- **§ 7.1** — the predicate scheme: first-match-wins, with multi-fires and zero-fires both routing to `other_ambiguous`.
- **§ 7.2** — the predicate order: symptom-attended → background-attended → lighting-attended → leaf-shape-attended.

Both decisions affect every category assignment in N04 and the resulting distribution in your paper.

### 7.1) The predicate scheme: first-match-wins with conjunction handling

For each heatmap, evaluate all four predicates and count how many fire. The assignment rule has three cases:

- **Exactly one fires.** Assign the heatmap to that category.
- **Two or more fire.** Assign the heatmap to `other_ambiguous`, and record which predicates matched in a `multi_fire_predicates` field for later inspection.
- **Zero fire.** Assign the heatmap to `other_ambiguous`, with all predicate booleans false.

"First-match-wins" describes the single-match case: when exactly one predicate fires, that's the category. The committed order in § 7.2 is what "first" refers to — the highest-priority predicate is the first to be checked, then the next, and so on. "Conjunction handling" is the multi-fire case: when more than one predicate fires, the conjunction (multiple categories applying simultaneously) is treated as ambiguous rather than resolved in favor of the highest-priority match.

**Zero-fires and multi-fires both route to `other_ambiguous`, but for opposite reasons.** A zero-fire heatmap doesn't meet any predicate's threshold — none of the four named failure modes describes what the model was doing. A multi-fire heatmap meets more than one threshold — multiple named failure modes describe what the model was doing simultaneously. Both are *outside* the four-category taxonomy, just from different directions. Your methods section should report both shares separately (zero-fire share and multi-fire share among "other" assignments) so a reader can tell whether the taxonomy is too narrow (lots of zero-fires) or too overlapping (lots of multi-fires).

**The alternative — letting multi-fires resolve in favor of the highest-priority predicate — was considered and rejected.** It would have given a cleaner-looking distribution (every heatmap that fires any predicate gets a named category) at the cost of hiding the ambiguity. With conjunction handling on, a non-trivial multi-fire share is a finding rather than an artifact: it means the taxonomy's named categories overlap in practice, which is something the paper should report explicitly rather than wash out into the most-prioritized matching category.

The locked commitment: `predicate_scheme = "first-match-wins, multi-fires route to other"`.

### 7.2) The predicate order

The four named predicates are ordered:

1. `symptom_attended_but_wrong_class`
2. `background_attended`
3. `lighting_attended`
4. `leaf_shape_attended`

(The five-category taxonomy has one more entry — `other_ambiguous` — but it doesn't appear here because it isn't a predicate; it's the destination for the multi-fire and zero-fire cases from § 7.1.)

Under the conjunction-handling scheme committed in § 7.1, the order doesn't drive single-fire assignments (only one predicate matches, so order is irrelevant) or zero-fire assignments (none match) — both cases are unambiguous. The order does two things instead.

**It documents the priority hierarchy the analyst intends.** If a future iteration of this methodology relaxes the conjunction handling — for example, to credit the highest-priority match when multiple fire — the order is the priority sequence that would apply. Committing it now means the priority has been thought about and recorded, not improvised later. A reader of the precommit who wants to know "if a heatmap had matched both symptom-attended and background-attended, which would the analyst have considered the more important finding?" can answer that question from the order field alone.

**It reflects how the named categories relate to the underlying question.** Symptom-attended-but-wrong-class goes first because it's the "honest mistake" category — the model was looking at the right kind of thing (a symptom on the leaf) but assigned the wrong disease. If symptom-attended fires for a heatmap, that's a meaningful finding about the model's class boundary, independent of whether the model was also paying some attention to the background or to lighting. Background-attended is second because Noyan's prior work makes it the most well-grounded of the remaining three. Lighting-attended is third — same direction as background-attended (an artifact of PV's collection conventions) but with thinner prior-work grounding. Leaf-shape-attended is last because it's the most ambiguous of the four: the leaf-shape signal is harder to isolate from "the model is attending to the leaf" generally, and the predicate is therefore the most likely to co-fire with other predicates rather than to be the single match.

The locked commitment: `predicate_order = ["symptom_attended_but_wrong_class", "background_attended", "lighting_attended", "leaf_shape_attended"]`.

In [ ]:
# Commit the predicate scheme and order (Section 7).
precommit["categorization_procedure"]["predicate_scheme"] = "first-match-wins, multi-fires route to other"
precommit["categorization_procedure"]["predicate_order"] = [
    "symptom_attended_but_wrong_class",
    "background_attended",
    "lighting_attended",
    "leaf_shape_attended",
]

# Print back what just got committed.
print(f"Predicate scheme: {precommit['categorization_procedure']['predicate_scheme']}")
print("Predicate order:")
for i, predicate in enumerate(precommit["categorization_procedure"]["predicate_order"], 1):
    print(f"  {i}. {predicate}")

## 8) Pre-commit your four threshold values

Each named predicate becomes a concrete numeric rule in this section. Five derived quantities get computed per heatmap in N04 (Section 4 of N04), and the four predicates are defined against subsets of them:

- **`m_out`** — the fraction of (normalized) attention mass falling *outside* the SAM-segmented leaf region. Used by T1.
- **`m_in`** — the fraction falling *inside* the leaf. Used by T4. (Note that `m_out` + `m_in` = 1 exactly, since the heatmap sums to 1 by construction and "outside leaf" + "inside leaf" partitions the pixel space.)
- **`m_perimeter`** — the fraction falling in a thin band around the leaf edge (band defined in § 8.2). Used by T2.
- **`spearman_corr_with_brightness`** — the pixelwise rank correlation between the heatmap and the image's brightness map. Used by T3.
- **`concentration`** — for in-leaf attention only: the fraction of in-leaf mass that falls in the smallest contiguous region capturing 80% of total in-leaf mass (definition in § 8.4). Used by T4.

Three of the four predicates threshold one derived quantity each (T1, T2, T3). The fourth — T4, symptom-attended-but-wrong-class — is a conjunction of two: the model has to be looking at the leaf AND looking at it in a focused way, both at once. The four threshold values follow:

- **§ 8.1** — T1 = 0.5 for `m_out` (background-attended).
- **§ 8.2** — T2 = 0.3 for `m_perimeter`, with the perimeter-band definition (leaf-shape-attended).
- **§ 8.3** — T3 = 0.5 for `|spearman_corr_with_brightness|` (lighting-attended).
- **§ 8.4** — T4 = (m_in ≥ 0.5 AND concentration > 0.6), with the concentration definition (symptom-attended-but-wrong-class).

These are working values for a Week-1 commit. The reasoning field at the `categorization_procedure` level (set in Section 5) is where you defend them against the obvious alternatives. What you commit to here is the locked starting point.

### 8.1) T1 = 0.5 for background-attended

The predicate fires when more than half the attention mass falls outside the leaf.

`background_attended := m_out > 0.5`

where `m_out` is the sum of (normalized) heatmap values at pixels outside the SAM mask. Under the ReLU-normalize transform from Section 6, `m_out` + `m_in` = 1 exactly, so T1 > 0.5 is equivalent to "more attention outside the leaf than inside it" — a strong claim about where the model was looking.

The 0.5 threshold is the "majority of attention" cutoff. A lower threshold (T1 = 0.4) would catch heatmaps with substantial but not majority outside-leaf attention; a higher threshold (T1 = 0.7) would require dominant outside-leaf attention. 0.5 is the natural choice for a Week-1 commit — it asks the question "did the model put more attention outside the leaf than on it?" with a yes/no answer.

The reasoning field at the categorization_procedure level (set in Section 5) is where you defend 0.5 against alternatives at submission. If your eventual taxonomy distribution shows zero or near-zero background-attended assignments, the threshold may have been too high; if it shows the majority, too low. The methods-section interpretation is part of what gets defended eventually — not in this notebook.

### 8.2) T2 = 0.3 for leaf-shape-attended

The predicate fires when more than 30% of attention mass concentrates in a thin band around the leaf's edge — the kind of pattern you'd see if the model is using the leaf outline as a shortcut for host identity.

`leaf_shape_attended := m_perimeter > 0.3`

where `m_perimeter` is the sum of (normalized) heatmap values inside the perimeter band, defined as:

**Perimeter band** = the SAM mask dilated by 8 pixels minus the SAM mask eroded by 8 pixels — a ring-shaped region straddling the leaf boundary, 16 pixels thick total (8 on each side).

The 0.3 threshold is lower than T1's 0.5 because the perimeter band is a much smaller region than "everything outside the leaf." For a heatmap upsampled from 7×7 to a typical PD image resolution, the perimeter band is usually a few percent of total pixels; if 30% of attention mass concentrates in a few percent of pixels, that's a strong concentration signal pointing at the edge specifically. Setting T2 = 0.5 would require attention to be majority-concentrated in the perimeter band, which is a sharper condition than necessary for the leaf-shape interpretation.

The 8-pixel dilation/erosion radius is a working choice for a Week-1 commit. A wider band (16 pixels) would catch broader edge-following patterns at the cost of bleeding into "general leaf attention"; a narrower band (4 pixels) would be more specific but might miss real edge-following due to Grad-CAM's coarse 7×7 origin. 8 pixels is the natural middle for a heatmap that started at 7×7 spatial resolution and got upsampled to roughly 224×224 or higher in N04's per-image processing.

### 8.3) T3 = 0.5 for lighting-attended

The predicate fires when the heatmap correlates strongly with the image's per-pixel brightness — either positively (model lighting up on bright regions) or negatively (model lighting up on dark regions).

`lighting_attended := abs(spearman_corr_with_brightness) > 0.5`

where `spearman_corr_with_brightness` is the rank correlation between the (flattened, normalized) heatmap values and the (flattened) per-pixel brightness map. The brightness map can be computed several ways — luminance from RGB (`0.2126·R + 0.7152·G + 0.0722·B`) or the V channel of HSV — and N04 settles the choice in its Section 4.5; the precommit doesn't pin it because the threshold conclusion is robust to either definition.

The absolute value is essential here. Lighting-attended is symmetric: a model that lights up on bright regions specifically (positive correlation) is using lighting as a class cue, and so is a model that lights up on dark regions specifically (negative correlation). Without `abs`, the predicate would only catch one direction.

The 0.5 threshold is the "strong correlation" cutoff. Spearman correlations above 0.5 are generally interpreted as substantial in the statistics literature; above 0.7 as strong. Setting T3 = 0.5 catches the substantial-to-strong range. A more conservative T3 = 0.7 would require near-monotonic agreement between heatmap and brightness; a more aggressive T3 = 0.3 would risk catching coincidental agreement on images where lighting happens to correlate with leaf-region structure (lighter centers, darker edges) for reasons unrelated to disease attention.

### 8.4) T4 = (m_in ≥ 0.5 AND concentration > 0.6) for symptom-attended-but-wrong-class

T4 is the only conjunction in the predicate set. Symptom-attended-but-wrong-class requires both that the model was looking at the leaf (majority of attention in-leaf, not outside) AND that the attention was focused in one region rather than diffuse across the leaf.

`symptom_attended_but_wrong_class := (m_in ≥ 0.5) AND (concentration > 0.6)`

**First conjunct: `m_in` ≥ 0.5.** The model has to be putting majority attention inside the leaf, not outside. Without this conjunct, an entirely background-attended heatmap could still pass the concentration check (a tight concentration of attention in a small background region) and be misassigned to symptom-attended. The `≥` rather than `>` is intentional: the 0.5 boundary is the symmetric partition with `m_out`, and a heatmap exactly at the boundary should qualify for the in-leaf interpretation rather than failing both T1 (m_out > 0.5 is strict) and T4 (m_in > 0.5 would also be strict), which would push the boundary case to a zero-fire outcome it doesn't really belong in.

**Second conjunct: `concentration` > 0.6.** This is the "focused, not diffuse" condition. To define it, first define the concentration quantity:

**Concentration** = a compactness measure of where in-leaf attention is concentrated, on a [0, 1] scale where 1 means tightly focused and 0 means spread across the whole leaf. The construction is: lower a threshold from the heatmap maximum downward until the in-leaf pixels above the threshold capture at least 80% of in-leaf attention mass; take the largest connected component of that set; then concentration = 1 − (component area / leaf area).

If the in-leaf attention is concentrated in one tight spot, the 80%-mass region is small, so (component area) / (leaf area) is small, and concentration is close to 1. If the in-leaf attention is diffuse — spread evenly across the whole leaf — the 80%-mass region has to cover most of the leaf to capture 80% of the mass, so (component area) / (leaf area) is close to 1, and concentration is close to 0. Concentration > 0.6 means the largest connected component of the 80%-mass region covers less than 40% of the leaf — focused enough to read as the model looking at a specific feature.

The 80% inner threshold for the 80%-mass region is a working choice — it's a standard cutoff for "where is most of the signal" in spatial-distribution analyses, and it gives a smooth concentration measure that doesn't depend heavily on outlier pixels (which a "capture 100%" version would). The 0.6 threshold on the resulting concentration is the cutoff between "focused enough to read as a single attended region" (above 0.6) and "spread out enough to read as general leaf attention" (below).

The combined predicate is asking: did the model put majority attention on the leaf, AND was that attention focused enough to look like the model was reading a specific feature rather than the leaf in general? If both, the failure mode is the model looking at the right kind of thing but assigning it to the wrong category — the "honest mistake" framing from Section 4.

In [ ]:
# Commit the four threshold values and supporting definitions (Section 8).
precommit["categorization_procedure"]["thresholds"] = {
    "T1_background_attended": {"m_out >": 0.5},
    "T2_leaf_shape_attended": {"m_perimeter >": 0.3},
    "T3_lighting_attended": {"abs(spearman_corr_with_brightness) >": 0.5},
    "T4_symptom_attended_but_wrong_class": {
        "m_in >=": 0.5,
        "concentration >": 0.6,
        "concentration_definition": "fraction of in-leaf attention in smallest contiguous region capturing 80% of in-leaf mass",
    },
    "perimeter_band_definition": "SAM mask dilated by 8 pixels minus SAM mask eroded by 8 pixels",
}

# Print back what just got committed.
thresholds = precommit["categorization_procedure"]["thresholds"]
t1 = thresholds["T1_background_attended"]["m_out >"]
t2 = thresholds["T2_leaf_shape_attended"]["m_perimeter >"]
t3 = thresholds["T3_lighting_attended"]["abs(spearman_corr_with_brightness) >"]
t4_min = thresholds["T4_symptom_attended_but_wrong_class"]["m_in >="]
t4_conc = thresholds["T4_symptom_attended_but_wrong_class"]["concentration >"]

print("Thresholds committed:")
print(f"  T1 (background-attended):                m_out > {t1}")
print(f"  T2 (leaf-shape-attended):                m_perimeter > {t2}")
print(f"  T3 (lighting-attended):                  |spearman_corr_with_brightness| > {t3}")
print(f"  T4 (symptom-attended-but-wrong-class):   m_in >= {t4_min} AND concentration > {t4_conc}")
print()
print("Supporting definitions:")
print(f"  Perimeter band: {thresholds['perimeter_band_definition']}")
print(f"  Concentration:  {thresholds['T4_symptom_attended_but_wrong_class']['concentration_definition']}")

## 9) Pre-commit your aggregation level

You'll report the failure-mode distribution at the overall taxonomy level — five numbers, one per category, summed across the entire misclassification set. Not per-class. Not per-host. The committed choice: `aggregation_level.choice = "overall_taxonomy_distribution"`.

The reason is sample size. PD's test split has roughly 236 images; the trained PV classifier from R2-Q1 gets roughly 80 of them wrong, spread across 13 host species and 4 disease categories. A per-class distribution would have on the order of 5–10 misclassifications per class — too few to support a "this class is mostly background-attended" claim with statistical confidence. Per-host doesn't help: 80 errors across 13 hosts gives roughly 6 errors per host, which collapses to the same problem. The overall distribution sidesteps the small-cell problem: 80 misclassifications categorized into five buckets gives bucket counts on the order of 5–30 each (depending on how the distribution shapes up), which is enough to read patterns from.

This commitment also constrains what counts as a reportable finding versus a hypothesis for the paper. Any apparent per-class pattern you notice while inspecting the heatmaps in N04 is a *hypothesis* for the paper's discussion section, not a *finding* for its results section. The results section reports the overall distribution; the discussion section is where speculation about per-class differences lives, with the statistical fragility flagged.

The R2-Q1 → R2-Q2 narrative bridge surfaced in Section 2 — *does the failure-mode distribution within healthy-misclassified images differ from the overall distribution?* — sits at the boundary of what this aggregation supports. The cross-category cut (healthy vs. fungal vs. bacterial vs. viral vs. pest) is per-class at the disease-category grain, with n ≈ 80/4 ≈ 20 per category — small but not impossibly so. The four-notebook structure handoff held this comparison for the paper rather than the notebooks; the notebooks report what the precommit says they report (overall distribution), and the paper has more latitude to make the comparison while acknowledging the statistical fragility explicitly.

In [ ]:
# Commit the aggregation level (Section 9).
precommit["aggregation_level"] = {
    "choice": "overall_taxonomy_distribution",
    "reasoning": (
        # Replace this placeholder with your own reasoning before submission.
        # Cover at minimum: why overall (the sample-size argument about ~80
        # misclassifications across 13 hosts and 4 categories); what apparent
        # per-class or per-host patterns you noticed in N04 and why those
        # belong in the paper's discussion section rather than its results
        # section; and how the cross-category cut from R2-Q1 (healthy vs
        # fungal etc.) sits at the boundary of what this aggregation supports.
        "PLACEHOLDER — REWRITE BEFORE SUBMISSION."
    ),
}

# Print back what just got committed.
print(f"Aggregation level: {precommit['aggregation_level']['choice']}")
print("Reasoning placeholder set at aggregation_level.")

## 10) Pre-commit your sanity-check criterion

Section 3.1 introduced what Adebayo's two sanity checks ARE — model-parameter randomization (progressively replace the trained model's weights with random values from the top layers backward and check that the saliency map diverges from the trained-model map) and data randomization (train a model from scratch on label-shuffled data and check that the resulting saliency map differs from the real model's). This section commits to HOW you'll quantify "diverges" and "differs": the metric, the passing criterion, and the commitment to running both checks rather than picking one.

### 10.1) Three commitments.

**The metric is Spearman rank correlation between two heatmaps.** Spearman ρ is the standard saliency-comparison metric in the literature — it's invariant to monotonic transformations (which matters because heatmap magnitudes vary across models and aren't directly comparable in absolute terms), and it works on the rank order of pixel attention values rather than the values themselves. Pearson correlation would be sensitive to magnitude scaling; structural similarity (SSIM) is more sensitive to spatial structure than to the rank order that "where did the model look?" depends on. Spearman captures the question that matters: does the same ranking of pixels-by-attention come out from the randomized model as from the trained model?

**The passing criterion is ρ < 0.3 at full model randomization.** "Full model randomization" means the trained model has been replaced layer-by-layer until the final block (`layer4` for ResNet-18) has random weights — the end state of the cascade Section 3.1 described. At that point, the saliency map of the randomized model should correlate weakly or not at all with the trained model's saliency map if the saliency method is reading what the model learned. ρ < 0.3 is the "weakly correlated" cutoff under the same Spearman-correlation interpretation conventions used in Section 8's T3 threshold — the opposite direction (T3 wants strong correlation to fire; this criterion wants weak correlation to pass). If ρ stays above 0.3 at full randomization, the saliency method is reading input-image patterns more than model parameters, and Grad-CAM's reliability on this trained model is in doubt.

**Both checks get run.** Model-parameter randomization AND data randomization. Not one or the other. Each catches a different failure mode of the saliency method — parameter randomization catches saliency methods that don't actually depend on model weights; data randomization catches saliency methods that produce stable heatmaps regardless of whether the model actually learned the task. Skipping one would leave the corresponding failure mode unchecked. The data-randomization check is expensive (it requires retraining a model on label-shuffled data, which the four-notebook structure handoff committed to running at full epoch count per R2-Q1's recipe), but the cost is what makes the check meaningful — a partial-epoch shuffled retrain wouldn't produce a model whose saliency map is interpretable as "no learned task."

N03 is where you execute both checks. This section just locks in the criterion they have to meet.

In [ ]:
# Commit the sanity-check criterion (Section 10).
precommit["sanity_checks"] = {
    "metric": "Spearman_rank_correlation",
    "passing_criterion": "rho < 0.3 at full model randomization",
    "checks_run": ["model_parameter_randomization", "data_randomization"],
    "reasoning": (
        # Replace this placeholder with your own reasoning before submission.
        # Cover at minimum: why Spearman rank correlation rather than Pearson
        # or SSIM (rank invariance, attention-magnitude scale insensitivity);
        # why rho < 0.3 specifically (Spearman-correlation interpretation
        # conventions, opposite direction from Section 8's T3 = 0.5 cutoff);
        # and why both Adebayo checks rather than just the cheaper
        # model-parameter randomization check.
        "PLACEHOLDER — REWRITE BEFORE SUBMISSION."
    ),
}

# Print back what just got committed.
sc = precommit["sanity_checks"]
print("Sanity-check criterion committed:")
print(f"  Metric:             {sc['metric']}")
print(f"  Passing criterion:  {sc['passing_criterion']}")
print(f"  Checks to run:      {', '.join(sc['checks_run'])}")
print("Reasoning placeholder set at sanity_checks level.")

## 11) Write `precommit.json`, sanity-load, summarize

The precommit dict is structurally complete after Section 10. Three operations close out the Week-1 deliverable:

1. **Write `precommit.json`** to `OUTPUT_DIR` so N02–N04 can load it.
2. **Sanity-load** the file back and assert that the round-trip matches the in-memory dict — catches any silent JSON-serialization issue while you're still in this notebook to fix it.
3. **Summarize** the committed decisions back to yourself: one screen, no JSON formatting, no scrolling through the file. The summary reads values from the `precommit` dict directly (not from the freshly-written file), so a typo at any step in Sections 4–10 would show up here as something unexpected rather than being hidden behind successful JSON serialization.

In [ ]:
# Write precommit.json to OUTPUT_DIR.
import json

precommit_path = OUTPUT_DIR / "precommit.json"
with open(precommit_path, "w") as f:
    json.dump(precommit, f, indent=2)

print(f"Wrote {precommit_path}")

# Sanity-load: read the file back and confirm round-trip.
with open(precommit_path, "r") as f:
    precommit_loaded = json.load(f)

assert precommit_loaded == precommit, \
    "Round-trip mismatch — precommit.json does not match the in-memory precommit dict."
print("Sanity-load passed: precommit.json round-trips to the in-memory dict.")

In [ ]:
# Verdict summary: read each committed value out of the precommit dict
# and print a flat, scannable readout of the Week-1 deliverable.
tax = precommit["taxonomy"]
proc = precommit["categorization_procedure"]
ls = proc["leaf_segmentation"]
thr = proc["thresholds"]
agg = precommit["aggregation_level"]
sc = precommit["sanity_checks"]

# Short labels for the four predicates (used in the predicate-order summary).
short = {
    "background_attended": "background",
    "leaf_shape_attended": "leaf-shape",
    "lighting_attended": "lighting",
    "symptom_attended_but_wrong_class": "symptom",
}
order_arrow = " → ".join(short[p] for p in proc["predicate_order"])

# Threshold values, pulled out as aliases for readability.
t1 = thr["T1_background_attended"]["m_out >"]
t2 = thr["T2_leaf_shape_attended"]["m_perimeter >"]
t3 = thr["T3_lighting_attended"]["abs(spearman_corr_with_brightness) >"]
t4_min = thr["T4_symptom_attended_but_wrong_class"]["m_in >="]
t4_conc = thr["T4_symptom_attended_but_wrong_class"]["concentration >"]

# Compact form for the SAM label.
sam_short = ls["method"].split(" ")[0]  # "SAM" from "SAM (Segment Anything Model)"

print("Your Week-1 pre-commits:")
print()
print(f"  Taxonomy:           {len(tax['categories'])} categories "
      f"({', '.join(tax['categories'])})")
print(f"  Categorization:     {proc['method']}")
print(f"  Leaf segmentation:  {sam_short}, {ls['prompting']}, "
      f"n={ls['validation']['n_validation_images']} validation, color fallback")
print(f"  Attention mass:     {proc['attention_mass_definition']}")
print(f"  Predicate scheme:   {proc['predicate_scheme']}")
print(f"  Predicate order:    {order_arrow}")
print(f"  Thresholds:         T1={t1}, T2={t2}, T3={t3}, "
      f"T4=(m_in>={t4_min} AND conc>{t4_conc})")
print(f"  Aggregation:        {agg['choice']}")
print(f"  Sanity check:       {sc['metric'].replace('_', ' ')}, {sc['passing_criterion']}")
print()
print("Reasoning placeholders still to rewrite (4):")
print("  - precommit['taxonomy']['reasoning']")
print("  - precommit['categorization_procedure']['reasoning']")
print("  - precommit['aggregation_level']['reasoning']")
print("  - precommit['sanity_checks']['reasoning']")
print()
print("Submit precommit.json as your Week-1 deliverable.")

## 12) Where to go from here

Week 1 is done. Your `precommit.json` is on disk in `OUTPUT_DIR` with the four top-level commitments — taxonomy, categorization procedure, aggregation level, sanity-check criterion. Before you submit, replace the four `reasoning` placeholders with your own defense of each commitment (Section 11's summary listed the paths). The Week 1 deliverable is the rewritten `precommit.json` plus EQ#1 per the workflow on the R2-Q2 question page.

N02 (Week 2) picks up with the load-and-filter work: it opens with a defensive load of `precommit.json`, loads the four R2-Q1 outputs catalogued in Section 2, filters the PD predictions to `correct_at_category == False`, and runs exploratory analysis on the resulting misclassification set. N03 (Week 3) runs Grad-CAM and the sanity checks; N04 (Week 4) applies your taxonomy and reports the distribution.

The commitments you made here aren't suggestions — they're what your methods section reports. When you open N02 next week, the first thing it does is read your `precommit.json` back. Anything you change between now and then changes the methodology, not just the result.